In [1]:
import sys
import warnings
from pathlib import Path
 
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, classification_report,
                             confusion_matrix, f1_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [3]:
warnings.filterwarnings("ignore")
src = '../prepared_data.parquet'
out = Path(sys.argv[2] if len(sys.argv) > 2 else "output")
(out / "charts").mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")
RED, GREY = "#E50914", "#564d4d"
SEED = 42

In [4]:
df = pd.read_parquet(src).reset_index(drop=True)
for c in ("type", "country", "rating"):
    df[c] = df[c].astype(str)

In [5]:
def to_list(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return []
    if isinstance(v, str):
        v = v.split(",")
    return [str(x).strip() for x in v if str(x).strip()]

In [ ]:
df["genres"] = df["listed_in"].apply(to_list)
df["n_genres"] = df["genres"].apply(len)
df["n_directors"] = df["director"].apply(to_list).apply(len)
df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month
df["lag_years"] = (df["year_added"] - df["release_year"]).clip(lower=0)
num = df["duration"].str.extract(r"(\d+)")[0].astype(float)
is_tv = df["type"].eq("TV Show")
df["runtime_min"] = np.where(is_tv, 0, num)
df["n_seasons"] = np.where(is_tv, num, 0)
df["is_tv"] = is_tv.astype(int)

In [7]:
TOP_COUNTRIES = df["country"].value_counts().head(20).index
country_oh = pd.get_dummies(df["country"].where(df["country"].isin(TOP_COUNTRIES), "Other"),prefix="country").astype(int)
genre_counts = df["genres"].explode().value_counts()
KEEP_GENRES = genre_counts[genre_counts >= 20].index
genre_oh = pd.DataFrame({f"genre_{g}": df["genres"].apply(lambda L, g=g: int(g in L))for g in KEEP_GENRES})

In [8]:
base_num = ["release_year", "year_added", "month_added", "lag_years", "n_genres", "n_directors"]
X_audience = pd.concat([df[base_num + ["runtime_min", "n_seasons", "is_tv"]], country_oh, genre_oh], axis=1)
X_type = pd.concat([df[base_num], country_oh], axis=1)

In [9]:
AUD = {"TV-MA": "Adult", "R": "Adult", "NC-17": "Adult",
       "TV-14": "Teen", "PG-13": "Teen",
       "TV-PG": "Family", "PG": "Family", "TV-G": "Family", "G": "Family",
       "TV-Y": "Family", "TV-Y7": "Family", "TV-Y7-FV": "Family"}
y_aud_all = df["rating"].map(AUD)                           # NR / UR -> NaN (unrated, dropped)
keep = y_aud_all.notna()
print(f"Audience task: {keep.sum()} titles ({(~keep).sum()} unrated NR/UR dropped)")
print(y_aud_all[keep].value_counts(), "\n")
print("Feature count - audience:", X_audience.shape[1], "| type:", X_type.shape[1])

Audience task: 5837 titles (78 unrated NR/UR dropped)
rating
Adult     2818
Teen      1854
Family    1165
Name: count, dtype: int64 

Feature count - audience: 58 | type: 27


In [10]:
def models(binary):
    cw = "balanced"
    return {
        "Logistic Regression": (make_pipeline(StandardScaler(),
                                LogisticRegression(max_iter=2000, class_weight=cw)),
                                {"logisticregression__C": [0.1, 1, 10]}),
        "Decision Tree": (DecisionTreeClassifier(class_weight=cw, random_state=SEED),
                          {"max_depth": [4, 8, 12], "min_samples_leaf": [5, 20]}),
        "Random Forest": (RandomForestClassifier(n_estimators=300, class_weight=cw,
                                                 random_state=SEED, n_jobs=2),
                          {"max_depth": [None, 14], "min_samples_leaf": [1, 3]}),
        "Gradient Boosting": (HistGradientBoostingClassifier(class_weight=cw, max_iter=200,
                                                             random_state=SEED),
                              {"learning_rate": [0.05, 0.1], "max_depth": [None, 4]}),
    }

In [12]:
def run_task(tag, X, y, labels, binary, pos=None, names=None):
    names = names or labels
    cv_metric = "average_precision" if binary else "f1_macro"
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
    print(f"\n===== {tag}: train {len(Xtr)} / test {len(Xte)} =====")
    skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
    rows, fitted = [], {}
 
    def score(name, est, cv_score):
        pred = est.predict(Xte)
        r = {"model": name, "cv_" + cv_metric: cv_score, "test_accuracy": accuracy_score(yte, pred),
             "test_macro_F1": f1_score(yte, pred, average="macro")}
        if binary:
            p = est.predict_proba(Xte)[:, list(est.classes_).index(pos)]
            r["test_ROC_AUC"] = roc_auc_score(yte == pos, p)
            r["test_PR_AUC"] = average_precision_score(yte == pos, p)
            r["test_F1_TV_Show"] = f1_score(yte, pred, pos_label=pos)
        rows.append(r)
 
    for name, strat in [("Baseline: majority class", "most_frequent"),
                        ("Baseline: random (class priors)", "stratified")]:
        d = DummyClassifier(strategy=strat, random_state=SEED).fit(Xtr, ytr)
        score(name, d, np.nan)
 
    for name, (est, grid) in models(binary).items():
        gs = GridSearchCV(est, grid, scoring=cv_metric, cv=skf, n_jobs=2).fit(Xtr, ytr)
        fitted[name] = gs.best_estimator_
        score(name, gs.best_estimator_, gs.best_score_)
        print(f"  {name:<22} cv {cv_metric}={gs.best_score_:.3f}  best={gs.best_params_}")
 
    res = pd.DataFrame(rows).set_index("model").round(3)
    print(res.to_string())
    res.to_csv(out / f"results_{tag}.csv")
 
    best = res.loc[list(fitted)].sort_values("cv_" + cv_metric, ascending=False).index[0]
    print(f"Best by cross-validation (not by test): {best}")
    est = fitted[best]
    pred = est.predict(Xte)
    rep = classification_report(yte, pred, labels=labels, target_names=names, digits=3)
    print(rep)
    (out / f"report_{tag}.txt").write_text(f"Best model: {best}\n\n{rep}")
 
    # confusion matrix (row-normalised) of the best model
    cm = confusion_matrix(yte, pred, labels=labels)
    cmn = cm / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(5.4, 4.6))
    sns.heatmap(cmn, annot=[[f"{cm[i, j]}\n({cmn[i, j]:.0%})" for j in range(len(labels))]
                            for i in range(len(labels))], fmt="", cmap="Reds", cbar=False,
                xticklabels=names, yticklabels=names, ax=ax)
    ax.set(title=f"{best}: confusion matrix (test)", xlabel="Predicted", ylabel="Actual")
    plt.tight_layout(); plt.savefig(out / f"charts/{tag}_confusion_matrix.png", dpi=150); plt.close()
 
    # model comparison
    cols = ["test_macro_F1", "test_accuracy"]
    cmp_ = res[cols].iloc[::-1]
    ax = cmp_.plot(kind="barh", figsize=(9, 4.8), color=[RED, GREY], width=0.75)
    ax.set(title=f"{tag}: model comparison on held-out test set", xlabel="Score", ylabel="")
    ax.set_xlim(0, 1)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
    plt.tight_layout(); plt.savefig(out / f"charts/{tag}_model_comparison.png", dpi=150); plt.close()
 
    # permutation importance on the test set (model-agnostic)
    pi = permutation_importance(est, Xte, yte, n_repeats=5, random_state=SEED, n_jobs=2,
                                scoring="f1_macro")
    imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False)
    imp.to_csv(out / f"importance_{tag}.csv", header=["macro_F1_drop"])
    top = imp.head(15)[::-1]
    fig, ax = plt.subplots(figsize=(8, 5.4))
    ax.barh(top.index, top.values, color=RED)
    ax.set(title=f"{best}: top features (permutation importance)", xlabel="Drop in macro-F1 when shuffled")
    plt.tight_layout(); plt.savefig(out / f"charts/{tag}_feature_importance.png", dpi=150); plt.close()
    return res, best, imp

In [13]:
res_a, best_a, imp_a = run_task("audience", X_audience[keep], y_aud_all[keep],
                                ["Adult", "Teen", "Family"], binary=False)
res_b, best_b, imp_b = run_task("type", X_type, is_tv.astype(int), [0, 1],
                                binary=True, pos=1, names=["Movie", "TV Show"])


===== audience: train 4669 / test 1168 =====
  Logistic Regression    cv f1_macro=0.644  best={'logisticregression__C': 0.1}
  Decision Tree          cv f1_macro=0.619  best={'max_depth': 8, 'min_samples_leaf': 20}
  Random Forest          cv f1_macro=0.655  best={'max_depth': None, 'min_samples_leaf': 3}
  Gradient Boosting      cv f1_macro=0.647  best={'learning_rate': 0.05, 'max_depth': 4}
                                 cv_f1_macro  test_accuracy  test_macro_F1
model                                                                     
Baseline: majority class                 NaN          0.483          0.217
Baseline: random (class priors)          NaN          0.390          0.344
Logistic Regression                    0.644          0.652          0.643
Decision Tree                          0.619          0.646          0.639
Random Forest                          0.655          0.676          0.668
Gradient Boosting                      0.647          0.664          0.656
Bes

In [14]:
dur_is_season = df["duration"].str.contains("Season").astype(int).to_frame("duration_is_seasons")
Xtr, Xte, ytr, yte = train_test_split(dur_is_season, df["type"], test_size=0.2,
                                      stratify=df["type"], random_state=SEED)
leak = DecisionTreeClassifier(max_depth=1).fit(Xtr, ytr)
print(f"\nLeakage check: the 'duration' unit alone (min vs Season) predicts Movie/TV with "
      f"accuracy {accuracy_score(yte, leak.predict(Xte)):.3f}")


Leakage check: the 'duration' unit alone (min vs Season) predicts Movie/TV with accuracy 1.000


In [15]:
Xa, ya = X_audience[keep], y_aud_all[keep]
Xa_tr, _, ya_tr, _ = train_test_split(Xa, ya, test_size=0.2, stratify=ya, random_state=SEED)
groups = {
    "genres": [c for c in Xa.columns if c.startswith("genre_")],
    "country": [c for c in Xa.columns if c.startswith("country_")],
    "runtime / seasons / type": ["runtime_min", "n_seasons", "is_tv"],
    "dates (release/added/lag)": ["release_year", "year_added", "month_added", "lag_years"],
    "counts (n_genres, n_directors)": ["n_genres", "n_directors"],
}
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced",
                            random_state=SEED, n_jobs=2)
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
from sklearn.model_selection import cross_val_score
full_cv = cross_val_score(rf, Xa_tr, ya_tr, cv=skf, scoring="f1_macro").mean()
abl = [{"feature_group_removed": "(none - all features)", "cv_macro_F1": full_cv, "change": 0.0}]
for g, cols in groups.items():
    sc = cross_val_score(rf, Xa_tr.drop(columns=cols), ya_tr, cv=skf, scoring="f1_macro").mean()
    abl.append({"feature_group_removed": g, "cv_macro_F1": sc, "change": sc - full_cv})
abl = pd.DataFrame(abl).round(3)
print("\nAblation (Random Forest, audience task):\n", abl.to_string(index=False))
abl.to_csv(out / "ablation_audience.csv", index=False)


Ablation (Random Forest, audience task):
          feature_group_removed  cv_macro_F1  change
         (none - all features)        0.655   0.000
                        genres        0.529  -0.126
                       country        0.636  -0.018
      runtime / seasons / type        0.643  -0.012
     dates (release/added/lag)        0.652  -0.003
counts (n_genres, n_directors)        0.651  -0.004


In [16]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
y_aud_all[keep].value_counts().reindex(["Adult", "Teen", "Family"]).plot(
    kind="bar", color=RED, ax=axes[0], rot=0, title="Audience level (rating group)")
df["type"].value_counts().plot(kind="bar", color=GREY, ax=axes[1], rot=0, title="Type")
for a in axes:
    a.set_xlabel(""); a.set_ylabel("Titles")
plt.tight_layout(); plt.savefig(out / "charts/class_distribution.png", dpi=150); plt.close()
print("\nDone. Outputs in", out.resolve())


Done. Outputs in D:\Python\netflix-recommendation-system-and-trend-projection\task 5\output
